<a href="https://colab.research.google.com/github/letsemalenao/Letsema-Lenao/blob/main/Letsema_Titanic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

titanic_path = kagglehub.competition_download('titanic')
letslen_kaggleinputtitanictrain_csv_path = kagglehub.dataset_download('letslen/kaggleinputtitanictrain-csv')
heptapod_titanic_path = kagglehub.dataset_download('heptapod/titanic')

print('Data source import complete.')


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session


In [ ]:
# Importing relevant dependencies
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

In [ ]:
train_data = pd.read_csv("/kaggle/input/competitions/titanic/train.csv")
train_data.head()

In [ ]:
test_data = pd.read_csv("/kaggle/input/competitions/titanic/test.csv")
test_data.head()

In [ ]:
#Check Missing Data
print(train_data.isnull().sum())

In [ ]:
#Check Missing Data
print(test_data.isnull().sum())

In [ ]:
#Median age
median_age = train_data['Age'].median()
print(f"\nImputing missing Age values with the median: {median_age:.2f}")

In [ ]:
#Fill missing Age values
train_data['Age'] = train_data['Age'].fillna(median_age)
test_data['Age'] = test_data['Age'].fillna(median_age)

**Feature Engineering—creating new meaningful features out of existing ones**
SibSp = number of siblings/spouses
Parch = number of parents/children
These can be combined into a single feature called FamilySize.

In [ ]:
#FEATURE ENGINEERING/ to get a new feature called Family size
train_data["FamilySize"] = train_data["SibSp"] + train_data["Parch"] + 1
test_data['FamilySize'] = test_data['SibSp'] + test_data['Parch'] + 1

# Is alone flag
train_data["IsAlone"] = (train_data["FamilySize"] == 1).astype(int)
test_data['IsAlone'] = np.where(test_data['FamilySize'] == 1, 1, 0)

**Feature Engineering: FamilySize and IsAlone**

This code performs feature engineering by creating two new features, FamilySize and IsAlone, from existing variables in the Titanic dataset. The FamilySize feature is calculated by combining the number of siblings/spouses aboard (SibSp) and the number of parents/children aboard (Parch), then adding 1 to include the passenger themselves. This creates a more meaningful feature that represents the total number of family

**Analyzing the new feature and its usefulness for machine learning**

In [ ]:
sns.countplot(x='FamilySize', data=train_data)
plt.title('Distribution of Family Size')
plt.show()

In [ ]:
family_survival = train_data.groupby('FamilySize')['Survived'].mean()

family_survival.plot(kind='bar')
plt.ylabel('Survival Rate')
plt.title('Survival Rate by Family Size')
plt.show()

In [ ]:
sns.barplot(x='IsAlone', y='Survived', data=train_data)
plt.title('Survival Rate: Alone vs Not Alone')
plt.show()

# **Further feature engineering experiments**

In [ ]:
#creating age categories
train_data['AgeGroup'] = pd.cut(
    train_data['Age'],
    bins=[0, 12, 19, 35, 60, 100],
    labels=['Child', 'Teen', 'YoungAdult', 'Adult', 'Senior']
)

test_data['AgeGroup'] = pd.cut(
    test_data['Age'],
    bins=[0, 12, 19, 35, 60, 100],
    labels=['Child', 'Teen', 'YoungAdult', 'Adult', 'Senior']
)
# pd.cut() splits continuous numerical values into categories.

In [ ]:
# Analysing age groups
sns.barplot(x='AgeGroup', y='Survived', data=train_data)
plt.title('Survival Rate by Age Group')
plt.xticks(rotation=45)
plt.show()

In [ ]:
# creating fare groups reflect economic statuse
train_data['FareGroup'] = pd.qcut(
    train_data['Fare'],
    q=4,
    labels=['Low', 'Medium', 'High', 'VeryHigh']
)

test_data['FareGroup'] = pd.qcut(
    test_data['Fare'],
    q=4,
    labels=['Low', 'Medium', 'High', 'VeryHigh']
)
# pd.cut() Creates bins using fixed ranges. pd.qcut() Creates bins with equal numbers of observations.

In [ ]:
# Analysing fare groups
sns.barplot(x='FareGroup', y='Survived', data=train_data)
plt.title('Survival Rate by Fare Group')
plt.show()

In [ ]:
#Categorical Encoding because machine learning needs numbers
encoder = LabelEncoder()

train_data['Sex'] = encoder.fit_transform(train_data['Sex'])
test_data['Sex'] = encoder.transform(test_data['Sex'])

In [ ]:
#Select Features to train the machine learning model
features = ['PassengerId', 'Pclass', 'Sex', 'Age', 'Fare',
            'FamilySize', 'IsAlone']
# use age group and fare group feature later
X = train_data[features]
y = train_data['Survived']

X_test_data_final = test_data[features]

In [ ]:
#Split the training data
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

# **Training The Models**

In [ ]:
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

pred_lr = lr.predict(X_valid)

print("Logistic Regression Accuracy:",
      accuracy_score(y_valid, pred_lr))

In [ ]:
dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)

pred_dt = dt.predict(X_valid)

print("Decision Tree Accuracy:",
      accuracy_score(y_valid, pred_dt))

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

pred_rf = rf.predict(X_valid)

print("Random Forest Accuracy:",
      accuracy_score(y_valid, pred_rf))

#  Comprehensive Model Evaluation

In [ ]:
# STEP: Comprehensive Model Evaluation

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve
)

import matplotlib.pyplot as plt
import seaborn as sns

# ---------------------------------------------------
# TRAIN THE MODEL
# ---------------------------------------------------

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)

# ---------------------------------------------------
# PREDICTIONS
# ---------------------------------------------------

# Predictions on training data
train_predictions = rf_model.predict(X_train)

# Predictions on validation/test data
valid_predictions = rf_model.predict(X_valid)

# Probability predictions for ROC-AUC
train_probabilities = rf_model.predict_proba(X_train)[:, 1]
valid_probabilities = rf_model.predict_proba(X_valid)[:, 1]

# ---------------------------------------------------
# TRAINING PERFORMANCE METRICS
# ---------------------------------------------------

print("========== TRAINING PERFORMANCE ==========\n")

train_accuracy = accuracy_score(y_train, train_predictions)
train_precision = precision_score(y_train, train_predictions)
train_recall = recall_score(y_train, train_predictions)
train_f1 = f1_score(y_train, train_predictions)
train_auc = roc_auc_score(y_train, train_probabilities)

print(f"Training Accuracy  : {train_accuracy:.4f}")
print(f"Training Precision : {train_precision:.4f}")
print(f"Training Recall    : {train_recall:.4f}")
print(f"Training F1-Score  : {train_f1:.4f}")
print(f"Training ROC-AUC   : {train_auc:.4f}")

# ---------------------------------------------------
# VALIDATION / TEST PERFORMANCE METRICS
# ---------------------------------------------------

print("\n========== VALIDATION PERFORMANCE ==========\n")

valid_accuracy = accuracy_score(y_valid, valid_predictions)
valid_precision = precision_score(y_valid, valid_predictions)
valid_recall = recall_score(y_valid, valid_predictions)
valid_f1 = f1_score(y_valid, valid_predictions)
valid_auc = roc_auc_score(y_valid, valid_probabilities)

print(f"Validation Accuracy  : {valid_accuracy:.4f}")
print(f"Validation Precision : {valid_precision:.4f}")
print(f"Validation Recall    : {valid_recall:.4f}")
print(f"Validation F1-Score  : {valid_f1:.4f}")
print(f"Validation ROC-AUC   : {valid_auc:.4f}")

# ---------------------------------------------------
# CLASSIFICATION REPORT
# ---------------------------------------------------

print("\n========== CLASSIFICATION REPORT ==========\n")

print(classification_report(y_valid, valid_predictions))

# ---------------------------------------------------
# CONFUSION MATRIX
# ---------------------------------------------------

cm = confusion_matrix(y_valid, valid_predictions)

plt.figure(figsize=(6,4))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Did Not Survive', 'Survived'],
    yticklabels=['Did Not Survive', 'Survived']
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

# ---------------------------------------------------
# ROC CURVE
# ---------------------------------------------------

fpr, tpr, thresholds = roc_curve(y_valid, valid_probabilities)

plt.figure(figsize=(6,4))
plt.plot(fpr, tpr, label=f"AUC = {valid_auc:.4f}")
plt.plot([0, 1], [0, 1], linestyle='--')

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

# ---------------------------------------------------
# OVERFITTING / UNDERFITTING ANALYSIS
# ---------------------------------------------------

print("\n========== MODEL FIT ANALYSIS ==========\n")

performance_gap = train_accuracy - valid_accuracy

print(f"Training Accuracy  : {train_accuracy:.4f}")
print(f"Validation Accuracy: {valid_accuracy:.4f}")
print(f"Performance Gap    : {performance_gap:.4f}")

# Detect overfitting
if performance_gap > 0.10:
    print("\nThe model is likely OVERFITTING.")
    print("Reason: Training performance is much higher than validation performance.")

# Detect underfitting
elif train_accuracy < 0.75 and valid_accuracy < 0.75:
    print("\nThe model is likely UNDERFITTING.")
    print("Reason: Both training and validation performance are low.")

# Good generalization
else:
    print("\nThe model appears to GENERALIZE reasonably well.")
    print("Training and validation scores are relatively close.")

# ---------------------------------------------------
# FEATURE IMPORTANCE
# ---------------------------------------------------

feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by='Importance',
    ascending=False
)

print("\n========== FEATURE IMPORTANCE ==========\n")
print(feature_importance)

# Plot feature importance
plt.figure(figsize=(8,5))

sns.barplot(
    x='Importance',
    y='Feature',
    data=feature_importance
)

plt.title("Feature Importance")
plt.show()